## Data Extraction From API Using Python

In [1]:
## Import necessary libraries
import pandas as pd
import numpy as np
import requests
import os, sys
from datetime import datetime

### 1. Fetching Live Stock Price

In [26]:
## URL for API request to fetch stock data for IBM at 5-minute intervals
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&outputsize=full&apikey=demo'

In [27]:
response = requests.get(url)

meta_data_dtl = {}
time_series_dtl = []

if response.status_code == 200:
    data = response.json()
    # print(data)
    # print(data.keys())

    ## Extracting Meta Data
    meta_data = data['Meta Data']

    ## Cleaning Meta Data Keys and Creating Dictionary
    for k, v in meta_data.items():
        key_name = k.split('. ')[1].replace(' ', '_').lower() ## Cleaning Key Names
        meta_data_dtl[key_name] = v ## Creating Key-Value Pairs in Dictionary
    
    ## Creating DataFrame for Meta Data
    meta_data_df = pd.DataFrame([meta_data_dtl]) ## Wrapping dictionary in a list to create single-row DataFrame
    meta_data_df.index = [0] ## Resetting index to 0
     
    ## Extracting Time Series Data
    time_series_data = data['Time Series (5min)']

    ## Cleaning Time Series Data and Creating List of Dictionaries
    for timestamp, values in time_series_data.items():
        time_series_dtl.append({
            'timestamp': timestamp,
            'open': values['1. open'],
            'high': values['2. high'],
            'low': values['3. low'],
            'close': values['4. close'],
            'volume': values['5. volume']
        })
    
    ## Creating DataFrame for Time Series Data
    time_series_df = pd.DataFrame(time_series_dtl)

    ## Converting Data Types
    pd.to_datetime(time_series_df['timestamp'])

In [33]:
## Data conversion for numerical columns
time_series_df[['open', 'high', 'low', 'close']] = time_series_df[['open', 'high', 'low', 'close']].astype(float)
time_series_df['volume'] = time_series_df['volume'].astype(int)

In [34]:
## Date Time conversion for timestamp column
time_series_df['timestamp'] = pd.to_datetime(time_series_df['timestamp'], format='%Y-%m-%d %H:%M:%S')

## 2. Fetching News Headlines Using News API

In [ ]:
api_key = '9cec6f0b91a8448aa792891830fea9c0'

url = f"https://newsapi.org/v2/top-headlines?country=us&category=business&apiKey={api_key}"
response = requests.get(url)

if response.status_code == 200:
    news_data = response.json()
    # print(news_data)
    # print(news_data.keys())

    articles_dtl = []

    # ## Extracting Articles Data
    articles = news_data['articles']

    ## Cleaning Articles Data and Creating List of Dictionaries
    for article in articles:
        # print(article)
        # print(article['source']['id'])
        # print(article['source']['name'])
        # print(article['author'])
        # print(article['title'])
        # print(article['description'])
        # print(article['url'])
        # print(article['publishedAt'])
        # print(article['content'])

        articles_dtl.append({
            'source_id': article['source']['id'],
            'source_name': article['source']['name'],
            'author': article['author'],
            'title': article['title'],
            'description': article['description'],
            'url': article['url'],
            'published_at': article['publishedAt'],
            'content': article['content']
        })
    
    ## Creating DataFrame for Articles Data
    articles_df = pd.DataFrame(articles_dtl)

In [55]:

## Date Time conversion for published_at column
articles_df['published_at'] = pd.to_datetime(articles_df['published_at'], format='%Y-%m-%dT%H:%M:%SZ')

In [56]:
articles_df.head(2)

,source_id,source_name,author,title,description,url,published_at,content
0,abc-news,ABC News,ABC News,Winning numbers drawn for $1.25 Powerball jack...,None,https://abcnews.go.com/US/powerball-jackpot-su...,2025-12-18 06:56:43,None
1,None,Tipranks.com,Shalu Saraf,Micron’s (MU) AI Story Defies the Sell-Off — W...,Micron Technology ($MU) jumped in after-hours ...,https://www.tipranks.com/news/microns-mu-ai-st...,2025-12-18 04:31:21,


In [57]:
articles_df.dtypes

source_id               object
source_name             object
author                  object
title                   object
description             object
url                     object
published_at    datetime64[ns]
content                 object
dtype: object

### Using Query Parameters in API Requests

In [ ]:
API_URL = "https://newsapi.org/v2/top-headlines"
API_KEY = "9cec6f0b91a8448aa792891830fea9c0"

params = {
    "country": "us",
    "apiKey": API_KEY,
    "pageSize": 1
}

articles_dtl = []
response = requests.get(API_URL, params=params)

if response.status_code == 200:
    data = response.json()

    df = data['articles']

    for article in df:
        articles_dtl.append({
            'source_id': article['source']['id'],
            'source_name': article['source']['name'],
            'author': article['author'],
            'title': article['title'],
            'description': article['description'],
            'url': article['url'],
            'url_to_image': article['urlToImage'],
            'published_at': article['publishedAt']
        })
else:
    raise Exception(f"API request failed with status code {response.status_code}")


In [74]:
## Creating DataFrame for Articles Data
articles_df1 = pd.DataFrame(articles_dtl)

## Data type information 
print("Data Types of DataFrame:")
print(articles_df1.dtypes)

## Size of DataFrame
print("\n")
print("Size of DataFrame:")
print(articles_df1.size)

## Displaying first 2 rows
print("\n")
print("First 2 Rows of DataFrame:")
articles_df1.head(2)

Data Types of DataFrame:
source_id       object
source_name     object
author          object
title           object
description     object
url             object
published_at    object
content         object
url_to_image    object
dtype: object


Size of DataFrame:
162


First 2 Rows of DataFrame:


,source_id,source_name,author,title,description,url,published_at,content,url_to_image
0,abc-news,ABC News,ABC News,Winning numbers drawn for $1.25 Powerball jack...,None,https://abcnews.go.com/US/powerball-jackpot-su...,2025-12-18T06:56:43Z,None,NaN
1,None,Tipranks.com,Shalu Saraf,Micron’s (MU) AI Story Defies the Sell-Off — W...,Micron Technology ($MU) jumped in after-hours ...,https://www.tipranks.com/news/microns-mu-ai-st...,2025-12-18T04:31:21Z,,NaN


In [76]:
## Date Time conversion for published_at column
articles_df1['published_at'] = pd.to_datetime(articles_df1['published_at'], format='%Y-%m-%dT%H:%M:%SZ')

In [79]:
## Displaying Data Types after conversion
print("After Date Time Conversion:")
print(articles_df1.dtypes)
print("\n")

## Displaying the DataFrames
print("Meta Data DataFrame:")
articles_df1.head(2)

After Date Time Conversion:
source_id               object
source_name             object
author                  object
title                   object
description             object
url                     object
published_at    datetime64[ns]
content                 object
url_to_image            object
dtype: object


Meta Data DataFrame:


,source_id,source_name,author,title,description,url,published_at,content,url_to_image
0,abc-news,ABC News,ABC News,Winning numbers drawn for $1.25 Powerball jack...,None,https://abcnews.go.com/US/powerball-jackpot-su...,2025-12-18 06:56:43,None,NaN
1,None,Tipranks.com,Shalu Saraf,Micron’s (MU) AI Story Defies the Sell-Off — W...,Micron Technology ($MU) jumped in after-hours ...,https://www.tipranks.com/news/microns-mu-ai-st...,2025-12-18 04:31:21,,NaN


## 3. Tracking the International Space Station (ISS) Location

In [83]:
import geocoder
import json
import urllib
import turtle
import time
import webbrowser

In [86]:
## URL for tracking ISS location
url = 'http://api.open-notify.org/astros.json'

## Opening the URL
response = urllib.request.urlopen(url)

if response.getcode() == 200:
    data = response.read()
    json_data = json.loads(data)

    print("Number of people in space:", json_data['number'])
    print("\n")

    for person in json_data['people']:
        print("Name:", person['name'])
        print("Craft:", person['craft'])
        print("\n")


Number of people in space: 12


Name: Oleg Kononenko
Craft: ISS


Name: Nikolai Chub
Craft: ISS


Name: Tracy Caldwell Dyson
Craft: ISS


Name: Matthew Dominick
Craft: ISS


Name: Michael Barratt
Craft: ISS


Name: Jeanette Epps
Craft: ISS


Name: Alexander Grebenkin
Craft: ISS


Name: Butch Wilmore
Craft: ISS


Name: Sunita Williams
Craft: ISS


Name: Li Guangsu
Craft: Tiangong


Name: Li Cong
Craft: Tiangong


Name: Ye Guangfu
Craft: Tiangong




In [91]:
file = open("iss_location.txt", "w")

file.write("Number of people in space: " + str(json_data['number']) + "\n\n")

for person in json_data['people']:
    file.write("Name: " + person['name'] + "\n")
    # file.write("Craft: " + person['craft'] + "\n\n")

file.close()

In [90]:
g = geocoder.ip('me')
print(g)
print(g.latlng)

<[OK] Ipinfo - Geocode [Bengaluru, Karnataka, IN]>
[12.9719, 77.5937]


In [93]:
screen = turtle.Screen()
screen.setup(width=720, height=360)
screen.setworldcoordinates(-180, -90, 180, 90)

screen.title("ISS Tracker")
screen.bgpic("world_map.gif")
iss = turtle.Turtle()
iss.shape("circle") 
iss.color("red")
iss.penup()

TclError: couldn't open "world_map.gif": no such file or directory